In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

==((====))==  Unsloth 2026.10.11: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.


In [ ]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

dataset = load_dataset('json', data_files={'train': 'train.jsonl', 'validation': 'validation.jsonl'})

from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3",
)

def formatting_prompts_func(batch):
    return { "text": tokenizer.apply_chat_template(batch["messages"], tokenize=False, add_generation_prompt=False) }

dataset = dataset.map(formatting_prompts_func, batched = True)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset = dataset["validation"],
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

Unsloth: Already have LoRA adapters! We shall skip this step.


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/25 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


... (4 outputs left out)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 200 | Num Epochs = 3 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,3.288926
2,3.219570
3,3.080251
4,2.951480
5,2.641498
6,2.024138
7,1.593802
8,1.179074
9,0.999671
10,0.815109


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


In [ ]:
model.save_pretrained_gguf(
    "jargaozero_model",
    tokenizer,
    quantization_method = "q4_k_m"
)

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in jargaozero_model/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [07:34<00:00, 113.63s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [06:45<00:00, 101.28s/it]


Unsloth: Merge process complete. Saved to `/content/jargaozero_model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b11985-mix-c116c32 (app-b11985-mix-c116c32-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['jargaozero_model_gguf/llama-3-8b-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...


Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.


Unsloth: All GGUF conversions completed successfully!
Generated files: ['jargaozero_model_gguf/llama-3-8b-Instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model jargaozero_model_gguf/llama-3-8b-Instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to jargaozero_model_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f jargaozero_model_gguf/Modelfile


{'save_directory': 'jargaozero_model',
 'gguf_directory': 'jargaozero_model_gguf',
 'gguf_files': ['jargaozero_model_gguf/llama-3-8b-Instruct.Q4_K_M.gguf'],
 'modelfile_location': 'jargaozero_model_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': True}

In [11]:
import json
from unsloth import FastLanguageModel

model = FastLanguageModel.for_inference(model)

test_samples = []
with open("test.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        test_samples.append(json.loads(line))

results = []
print(f"Iniciando avaliação de {len(test_samples)} amostras de teste...\n")

for i, sample in enumerate(test_samples):
    messages = sample["messages"]

    user_prompt = [m for m in messages if m["role"] == "user"][0]["content"]
    reference_output = [m for m in messages if m["role"] == "assistant"][0]["content"]

    input_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_prompt}],
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

    outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True, temperature=0.1)
    generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    results.append({
        "id": i + 1,
        "prompt": user_prompt,
        "reference": reference_output,
        "model_output": generated_text
    })
    print(f"Amostra {i+1}/{len(test_samples)} processada.")

with open("evaluation_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("\nAvaliação concluída! Resultados salvos em 'evaluation_results.json'.")

Iniciando avaliação de 25 amostras de teste...



Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Amostra 1/25 processada.


Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Amostra 2/25 processada.


Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


... (39 outputs left out)

Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Amostra 23/25 processada.


Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Amostra 24/25 processada.
Amostra 25/25 processada.

Avaliação concluída! Resultados salvos em 'evaluation_results.json'.


In [12]:
import json

with open("evaluation_results.json", "r", encoding="utf-8") as f:
    results = json.load(f)

for item in results[:3]:
    print(f"--- Amostra ID: {item['id']} ---")
    print(f"Prompt / Entrada: {item['prompt']}")
    print(f"Referência (Esperado): {item['reference']}")
    print(f"Modelo Treinado (Gerado): {item['model_output']}")
    print("="*60 + "\n")

--- Amostra ID: 1 ---
Prompt / Entrada: Nível: Intermediário Técnico. Texto: Resource Quota Limit refere-se a quê?
Referência (Esperado): Em termos simples, resource quota limit refere-se a: Restrição máxima de consumo de recursos computacionais alocados a um projeto ou equipe. É um conceito fundamental na engenharia de sistemas para garantir eficiência e estabilidade operacional.
Modelo Treinado (Gerado): Em termos simples, resource quota limit refere-se a: Limite máximo de consumo de recursos computacional que um projeto ou equipe pode utilizar dentro de um ambiente de nuvem. É um conceito fundamental na engenharia de sistemas para garantir eficiência e estabilidade operacional.

--- Amostra ID: 2 ---
Prompt / Entrada: Nível: Leigo. Texto: Poderia explicar o termo técnico 'Bandwidth Throttling'?
Referência (Esperado): Em termos simples, bandwidth throttling refere-se a: Limitação intencional da velocidade da banda de internet por um provedor ou sistema. É um conceito fundamental na e

In [13]:
import gradio as gr
from unsloth import FastLanguageModel

def traduzir_jargao(texto_usuario):
    prompt_texto = f"Nível: Leigo. Texto: Poderia explicar o termo técnico '{texto_usuario}'?"

    input_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt_texto}],
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True, temperature=0.1)
    resposta = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    return resposta

demo = gr.Interface(
    fn=traduzir_jargao,
    inputs=gr.Textbox(label="Insere o Termo Técnico (ex: Resource Quota Limit)"),
    outputs=gr.Textbox(label="Tradução e Explicação do JargãoZero"),
    title="JargãoZero - Tradutor de Terminologia Tecnológica",
    description="Insere um termo técnico complexo para ver a tradução otimizada pelo modelo fine-tuned."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://11d966c5fe4d0d685e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [14]:
import json

with open("evaluation_results.json", "r", encoding="utf-8") as f:
    results = json.load(f)

total = len(results)
exact_matches = 0
partial_matches = 0

for item in results:
    ref = item["reference"].strip().lower()
    gen = item["model_output"].strip().lower()

    if ref == gen:
        exact_matches += 1
    elif any(word in gen for word in ref.split()[:5]):
        partial_matches += 1

exact_match_score = (exact_matches / total) * 100
accuracy_score = ((exact_matches + partial_matches) / total) * 100

print(f"=== RESULTADOS DA AVALIAÇÃO QUANTITATIVA ===")
print(f"Total de amostras testadas: {total}")
print(f"Exact Match (Correspondência Exata): {exact_match_score:.2f}%")
print(f"Acurácia/Semelhança Semântica: {accuracy_score:.2f}%")

=== RESULTADOS DA AVALIAÇÃO QUANTITATIVA ===
Total de amostras testadas: 25
Exact Match (Correspondência Exata): 20.00%
Acurácia/Semelhança Semântica: 100.00%
